In [ ]:
"""
ML Service - FastAPI application for ERIS ML predictions
"""
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from contextlib import asynccontextmanager
import logging
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Configure logging
logging.basicConfig(
    level=os.getenv("LOG_LEVEL", "INFO"),
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
@asynccontextmanager
async def lifespan(app: FastAPI):
    """Lifespan context manager for startup and shutdown events"""
    # Startup
    logger.info("Starting ML Service...")
    logger.info("Initializing Redis connection...")
    logger.info("Loading ML models...")
    
    yield
    
    # Shutdown
    logger.info("Shutting down ML Service...")
    logger.info("Closing Redis connection...")

In [ ]:
# Create FastAPI app
app = FastAPI(
    title="ERIS ML Service",
    description="Machine Learning service for Emergency Response Intelligence System",
    version="1.0.0",
    lifespan=lifespan
)

In [ ]:
# Configure CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Configure appropriately for production
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [ ]:
@app.get("/")
async def root():
    """Root endpoint"""
    return {
        "service": "ERIS ML Service",
        "version": "1.0.0",
        "status": "running"
    }

In [ ]:
@app.get("/health")
async def health_check():
    """Health check endpoint"""
    return {
        "status": "healthy",
        "redis": "connected",  # TODO: Check actual Redis connection
        "models": "loaded"  # TODO: Check actual model status
    }

In [ ]:
# Include routers — wrapped so a broken router doesn't kill the whole service
try:
    from ml_service.routers import data_generation
    app.include_router(data_generation.router)
except Exception as e:
    logger.warning(f"data_generation router failed to load: {e}")

In [ ]:
try:
    from ml_service.routers import features
    app.include_router(features.router)
except Exception as e:
    logger.warning(f"features router failed to load: {e}")

In [ ]:
try:
    from ml_service.routers import predictions
    app.include_router(predictions.router)
except Exception as e:
    logger.warning(f"predictions router failed to load: {e}")

In [ ]:
try:
    from ml_service.routers import quality
    app.include_router(quality.router)
except Exception as e:
    logger.warning(f"quality router failed to load: {e}")

In [ ]:
try:
    from ml_service.routers import monitoring
    app.include_router(monitoring.router)
except Exception as e:
    logger.warning(f"monitoring router failed to load: {e}")

In [ ]:
# Error handlers
@app.exception_handler(HTTPException)
async def http_exception_handler(request, exc):
    logger.error(f"HTTP error: {exc.detail}")
    return {
        "error": exc.detail,
        "status_code": exc.status_code
    }

In [ ]:
@app.exception_handler(Exception)
async def general_exception_handler(request, exc):
    logger.error(f"Unexpected error: {str(exc)}", exc_info=True)
    return {
        "error": "Internal server error",
        "detail": str(exc),
        "status_code": 500
    }

In [ ]:
if __name__ == "__main__":
    import uvicorn

    port = int(os.getenv("ML_SERVICE_PORT", 8000))
    host = os.getenv("ML_SERVICE_HOST", "0.0.0.0")

    uvicorn.run(
        "ml_service.app:app",
        host=host,
        port=port,
        reload=True,
        log_level=os.getenv("LOG_LEVEL", "info").lower()
    )